**Data Storytelling: Analysing Survival on the Titanic**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# sns plot style
sns.set(style='whitegrid')

# Loading & understanding Data

In [ ]:
!git clone 'https://github.com/GeeksforgeeksDS/21-Days-21-Projects-Dataset'

In [ ]:
df = pd.read_csv('/content/21-Days-21-Projects-Dataset/Datasets/Titanic-Dataset.csv')

print("Top 5 rows:")
df.head()

In [ ]:
print("Bottom 5 rows:")
df.tail()

## *Metadata of dataset*

In [ ]:
print("R x C")
df.shape

In [ ]:
print("Information of Dataset\n")
df.info()

print("\nDescription of dataset\n")
df.describe()

# Data Cleaning

In [ ]:
print("check total null values per feature")
df.isna().sum()

1st, Age is numerical - fill missing values with median

In [ ]:
print("Median of Age: ")
median_age = df['Age'].median()
print(median_age)

In [ ]:
df['Age'] = df['Age'].fillna(median_age)

print("Age feature after no null values")
print(df['Age'])

In [ ]:
print(df.isna().sum())

# Encoding Cabin feature to Has_Cabin to 0-no cabin & 1-has cabin as Cabin feature has a lot of missing values

In [ ]:
df['Has_Cabin'] = df['Cabin'].notna().astype(int)
print(df)

In [ ]:
df.drop('Cabin', axis=1, inplace=True)
print(df)

In [ ]:
print("Frequency count for categories of Has_Cabin\n")
df['Has_Cabin'].value_counts()

# Embarked Feature has only 2 missing values, filling using mode

In [ ]:
embarked_mode = df['Embarked'].mode()[0]
print("Mode of embarked feature is: ", end="")
print(embarked_mode)

In [ ]:
df['Embarked'] = df['Embarked'].fillna(embarked_mode)
print("Embarked Feature after removing na")
df['Embarked']

In [ ]:
print("Check if any missing value still left in an feature")
df.isna().sum()

## Univariate data analysis

Categorical vars - bar charts, piecharts.
Numerical vars - histogram, kd plots

In [ ]:
print("Categorical feature analysis")
fig, axes = plt.subplots(2, 3, figsize=(18,12))
fig.suptitle('Univariate Analysis of Categorical Features', fontsize=16)

sns.countplot(ax=axes[0, 0], x='Survived', data=df).set_title('Survival Distribution')
sns.countplot(ax=axes[0, 1], x='Pclass', data=df).set_title('Passenger Class Distribution')
sns.countplot(ax=axes[0, 2], x='Sex', data=df).set_title('Gender Distribution')
sns.countplot(ax=axes[1, 0], x='Embarked', data=df).set_title('Port of Embarkation')
sns.countplot(ax=axes[1, 1], x='SibSp', data=df).set_title('Siblings/Spouses Aboard')
sns.countplot(ax=axes[1, 2], x='Parch', data=df).set_title('Parents/Children Aboard')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
print(df['Parch'].value_counts())

In [ ]:
print("\nAnalyzing numerical features:")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Univariate Analysis of Numerical Features', fontsize=16)
sns.histplot(ax=axes[0], data=df, x='Age', kde=True, bins=30).set_title('Age Distribution')
sns.histplot(ax=axes[1], data=df, x='Fare', kde=True, bins=40).set_title('Fare Distribution')

plt.show()

## Bivariate data analysis

In [ ]:
print("Bivariate Analysis: Feature vs. Survival")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Bivariate Analysis with Survival', fontsize=16)
sns.barplot(ax=axes[0, 0], x='Pclass', y='Survived', data=df).set_title('Survival Rate by Pclass')
sns.barplot(ax=axes[0, 1], x='Sex', y='Survived', data=df).set_title('Survival Rate by Sex')
sns.barplot(ax=axes[1, 0], x='Embarked', y='Survived', data=df).set_title('Survival Rate by Port')
sns.barplot(ax=axes[1, 1], x='Has_Cabin', y='Survived', data=df).set_title('Survival Rate by Cabin Availability')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
facet_grid = sns.FacetGrid(df, col='Survived', height=6)
facet_grid.map(sns.histplot, 'Age', bins=25, kde=True)
plt.suptitle('Age Distribution by Survival Status', y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.boxplot(y='Fare', data=df)
plt.title('Box Plot of Ticket Fare')
plt.ylabel('Fare')
plt.show()

# Feature Engineering

In [ ]:
# 1. Create a 'FamilySize' feature
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# 2. Create an 'IsAlone' feature
df['IsAlone'] = 0
df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

print("Created 'FamilySize' and 'IsAlone' features:")
df[['FamilySize', 'IsAlone']].head()

In [ ]:
# Analyze the new family-related features against survival
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Survival Rate by FamilySize
sns.barplot(ax=axes[0], x='FamilySize', y='Survived', data=df).set_title('Survival Rate by Family Size')

# Survival Rate by IsAlone
sns.barplot(ax=axes[1], x='IsAlone', y='Survived', data=df).set_title('Survival Rate for Those Traveling Alone')

plt.show()

In [ ]:
# 3. Extract 'Title' from the 'Name' column
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

print("Extracted Titles:")
df['Title'].value_counts()

In [ ]:
# Simplify the titles by grouping rare ones into a 'Rare' category
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')

df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# Let's see the survival rate by the new, cleaned titles
plt.figure(figsize=(12, 6))
sns.barplot(x='Title', y='Survived', data=df)
plt.title('Survival Rate by Title')
plt.ylabel('Survival Probability')
plt.show()

## Multivariate data analysis

In [ ]:
sns.catplot(x='Pclass', y='Survived', hue='Sex', data=df, kind='bar', height=6, aspect=1.5)
plt.title('Survival Rate by Pclass and Sex')
plt.ylabel('Survival Probability')
plt.show()

In [ ]:
# Violin plot

plt.figure(figsize=(14, 8))
sns.violinplot(x='Sex', y='Age', hue='Survived', data=df, split=True, palette={0: 'blue', 1: 'orange'})
plt.title('Age Distribution by Sex and Survival')
plt.show()

# Correlation Analsis

In [ ]:
plt.figure(figsize=(14, 10))
numeric_cols = df.select_dtypes(include=np.number)
print(numeric_cols)

correlation_matrix = numeric_cols.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

In [ ]:
!pip install ydata-profiling -q

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Titanic Dataset Y-Data Profiling Report")
profile.to_notebook_iframe()

In [ ]:
profile.to_file("Final_Report-Project1_EDA_Titanic-Survival-Dataset.html")